![Enterprise Backends](assets/banner-06.svg)

Many companies already run Datadog, Dynatrace, New Relic or Grafana for their infrastructure. The question is rarely "which LLM tool", it is "how do AI traces land next to everything else, without locking the code to one vendor?"

The answer is the **OpenTelemetry Collector**: a separate process that receives telemetry, transforms it, and sends it wherever you configure. In this notebook a real Collector runs in Docker, scrubs sensitive fields, and forwards to Langfuse. Adding Datadog or Dynatrace is a few lines of YAML.

## The architecture

```text
 weather agent ──OTLP──▶  OpenTelemetry Collector  ──▶ Langfuse      (running locally)
 (any language)           ├ attributes/scrub         ──▶ debug log     (running locally)
                          ├ batch                    ┄┄▶ Datadog       (config ready)
                          └ ...                      ┄┄▶ Dynatrace     (config ready)
                                                     ┄┄▶ LangSmith     (config ready)
```

Why teams put a Collector in the middle:

| Benefit | How |
|---|---|
| **One place for credentials** | Apps send to the Collector; only it holds vendor API keys |
| **Privacy at the edge** | Hash user ids, delete prompt text, drop attributes before data leaves your network |
| **Fan out** | Same spans to an LLM tool for engineers and an APM for SRE |
| **Sampling and cost control** | Tail sampling keeps every error and slow trace, samples the rest |
| **Vendor changes without deploys** | Edit YAML, restart the Collector, apps untouched |

## 1. Start the Collector

The config lives in `collector/otel-collector.yaml`. Here is the part that matters:

In [1]:
from pathlib import Path

config = (Path.cwd().parent / "collector" / "otel-collector.yaml").read_text()
print(config[config.index("processors:") : config.index("exporters:")])

processors:
  # Privacy at the edge: the app can be sloppy, the data that leaves still is not.
  attributes/scrub:
    actions:
      - key: user.id
        action: hash            # keep per-user analytics, drop the raw email
      - key: gen_ai.input.messages
        action: delete          # never ship raw prompts
      - key: input.value
        action: delete
      - key: deployment.environment
        value: workshop
        action: upsert
  batch:
    timeout: 2s




In [2]:
import subprocess
import time

collector_dir = Path.cwd().parent / "collector"
subprocess.run(["docker", "compose", "up", "-d"], cwd=collector_dir, capture_output=True, text=True)
time.sleep(3)
status = subprocess.run(
    ["docker", "compose", "ps", "--format", "{{.Service}}: {{.Status}}"], cwd=collector_dir, capture_output=True, text=True
).stdout
print(status.strip(), "\nlistening on localhost:4318 (OTLP/HTTP) and localhost:4317 (OTLP/gRPC)")

otel-collector: Up 5 minutes 
listening on localhost:4318 (OTLP/HTTP) and localhost:4317 (OTLP/gRPC)


## 2. Point the app at the Collector

The application only knows one address, `localhost:4318`. No vendor names, no API keys. To make the privacy step visible, the app is careless on purpose: it puts a raw email in `user.id` and the full prompt in `input.value`.

In [3]:
import lab
from openai import OpenAI
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.trace import SpanKind

provider = TracerProvider(resource=Resource.create({"service.name": "weather-agent-via-collector"}))
provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter(endpoint="http://localhost:4318/v1/traces")))
tracer = provider.get_tracer("weather-agent")
client = OpenAI()
MODEL = "gpt-4o-mini"
EMAIL = "priya.sharma@example.com"


def answer_question(question: str, city: str) -> str:
    with tracer.start_as_current_span("invoke_agent weather-agent") as root:
        root.set_attribute("gen_ai.operation.name", "invoke_agent")
        root.set_attribute("user.id", EMAIL)
        root.set_attribute("input.value", question)
        with tracer.start_as_current_span("execute_tool get_weather") as span:
            span.set_attribute("gen_ai.operation.name", "execute_tool")
            span.set_attribute("gen_ai.tool.name", "get_weather")
            weather = lab.get_weather_impl(city, flaky=False)
        with tracer.start_as_current_span(f"chat {MODEL}", kind=SpanKind.CLIENT) as span:
            span.set_attribute("gen_ai.operation.name", "chat")
            span.set_attribute("gen_ai.provider.name", "openai")
            span.set_attribute("gen_ai.request.model", MODEL)
            messages = lab.build_messages(question, [], weather)
            span.set_attribute("input.value", messages[-1]["content"])
            r = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=60)
            span.set_attribute("gen_ai.usage.input_tokens", r.usage.prompt_tokens)
            span.set_attribute("gen_ai.usage.output_tokens", r.usage.completion_tokens)
            return r.choices[0].message.content


print(answer_question(f"Hi, I'm {EMAIL}. Umbrella in London?", "London"))
provider.force_flush();

Yes, it's advisable to bring an umbrella in London as there's an 85% chance of rain.


## 3. Watch the Collector work

The `debug` exporter logs every batch it forwards.

In [4]:
time.sleep(4)
logs = subprocess.run(["docker", "compose", "logs", "--since", "30s"], cwd=collector_dir, capture_output=True, text=True).stdout
import json

for line in logs.strip().splitlines():
    if "\tTraces\t" in line:
        stats = json.loads(line[line.index("{") :])
        print(f"{line.split()[2][:19]}  forwarded {stats['spans']} spans from {stats['resource spans']} service")

2026-10-21T00:34:11  forwarded 3 spans from 1 service


## 4. Check what reached the backend

Langfuse stores the root span's attributes. The raw email and the prompt never arrived; the user id is a stable hash, so per-user analytics still work.

In [5]:
import hashlib

import pandas as pd

time.sleep(3)
recent = lab.langfuse_observations(limit=20, name="invoke_agent weather-agent", fields="core,basic,metadata,io")
root = recent.sort_values("startTime").iloc[-1]
blob = str(root.to_dict())

pd.DataFrame(
    [
        {"check": "raw email present anywhere", "result": EMAIL in blob},
        {"check": "prompt text present", "result": "Umbrella in London" in blob},
        {"check": "user id stored", "result": root.get("userId")},
        {"check": "user id equals SHA-256 of the email", "result": root.get("userId") == hashlib.sha256(EMAIL.encode()).hexdigest()},
        {"check": "environment tag added by collector", "result": "workshop" in blob},
    ]
).style.hide(axis="index")

check,result
raw email present anywhere,False
prompt text present,False
user id stored,6cd1cf87520cfbc7e19f29ac8b858b773765202593cef5958de192930e121f34
user id equals SHA-256 of the email,True
environment tag added by collector,True


The Collector's `hash` action uses SHA-256, so the stored user id is the SHA-256 of the email. The same person always maps to the same id, but the id cannot be read back as an email.

## 5. Adding Datadog, Dynatrace or LangSmith

Each is an extra exporter plus one entry in the pipeline. The commented blocks in `collector/otel-collector.yaml` are ready to enable:

```yaml
exporters:
  otlphttp/dynatrace:
    endpoint: https://${env:DT_ENV_ID}.live.dynatrace.com/api/v2/otlp
    headers:
      Authorization: "Bearer ${env:DT_PLATFORM_TOKEN}"   # scope: openpipeline:traces:ingest

  otlphttp/datadog:
    traces_endpoint: ${env:DD_OTLP_TRACES_ENDPOINT}     # site-specific, from Datadog docs
    headers:
      dd-api-key: ${env:DD_API_KEY}
      dd-otlp-source: llmobs                            # routes spans to LLM Observability

service:
  pipelines:
    traces:
      exporters: [otlphttp/langfuse, otlphttp/dynatrace, otlphttp/datadog, debug]
```

## 6. Vendor SDKs, when you want more

OTel gets you portable traces. Vendor SDKs add product-specific features. Datadog's is a good example:

```bash
pip install ddtrace
DD_LLMOBS_ENABLED=1 DD_LLMOBS_ML_APP=weather-agent \
DD_API_KEY=<key> DD_SITE=datadoghq.com DD_LLMOBS_AGENTLESS_ENABLED=1 \
ddtrace-run python app.py
```

```python
from ddtrace.llmobs.decorators import agent, tool, retrieval

@tool
def get_weather(city): ...

@agent
def answer_question(question): ...
```

`ddtrace-run` auto-instruments OpenAI, Anthropic, Bedrock and LangChain. Datadog span kinds are `llm`, `workflow`, `agent`, `tool`, `task`, `embedding` and `retrieval`.

## 7. The landscape at a glance

| Platform | Type | Strengths | Ingest |
|---|---|---|---|
| **Langfuse** | Open source LLM engineering | Self-hosting, tracing, prompts, evals, cost | SDK, OTLP |
| **LangSmith** | Managed LLM platform | LangChain/LangGraph depth, evals, annotation | SDK, OTLP |
| **Arize Phoenix** | Open source LLM observability | Notebook-friendly, OpenInference, evals | OTLP |
| **Datadog LLM Observability** | APM add-on | Next to infra and APM, security scanning, cluster insights | `ddtrace`, OTLP (GenAI semconv 1.37+) |
| **Dynatrace** | APM platform | Full-stack topology, AI-driven root cause, business metrics | OTLP, OpenLLMetry |
| **Grafana (Tempo)** | Open source stack | Cheap trace storage, dashboards you already have | OTLP |

## Choosing

- **Data must stay inside**: self-host Langfuse or Phoenix.
- **LangChain or LangGraph everywhere**: LangSmith.
- **The company runs Datadog or Dynatrace**: use their LLM features, fed by OTel.
- **Not sure yet**: instrument with OpenTelemetry, run a Collector, and try two backends side by side. The one your team opens every day wins.

## Cleanup

```bash
docker compose -f collector/docker-compose.yml down
docker compose -f langfuse/docker-compose.yml down      # add -v to delete data
```